In [1]:

import sympy as sp
from sympy.physics.mechanics import ReferenceFrame, dynamicsymbols
import sympy.physics.vector as spv
from sympy.physics.vector.printing import init_vprinting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import display, Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
init_vprinting(use_latex="mathjax")

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return ReferenceFrame(
        frame,
        latexs=(
            rf"\; {{}}^\mathcal {frame} \hat {x}",
            rf"\;{{}}^\mathcal {frame} \hat {y}",
            rf"\: {{}}^\mathcal {frame} \hat {z}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

# Problem Set 4

[Problem Set 4](./MIT8_01F16_pset4.pdf)

## Problem 4.1

![Problem 4.1](../figures/PS0401-Pulley-System.jpg)

A block of mass $m_1$ is at rest on an inclined plane that makes an angle $\theta$ with the
horizontal. The coefficient of static friction between the block and the incline surface
is $\mu_s$. A massless, inextensible string is attached to one end of the block, passes over a
fixed pulley, pulley 1, around a second freely suspended pulley, pulley 2, and is finally
attached to a fixed support. The left hand part of the string is parallel to the surface
of the inclined plane. The sections of the string coming off the suspended pulley are
vertical. The pulleys are massless, but a second block of mass $m_2$ is hung from the
suspended pulley. Gravity acts downward.

(a) What is $m_{2,\min}$, the minimum value of $m_2$ for which block 1 just barely slides up
the incline? Express your answers in terms of some or all of the variables $m_1$, $\theta$,
$\mu_s$ and $g$.

(b) What is $m_{2,\max}$, the maximum value of $m_2$ for which block 1 just barely slides
down the incline? Express your answers in terms of some or all of the variables
$m_1$, $\theta$, $\mu_s$ and $g$.

(c) Now assume that the block on the incline plane is sliding upward. The coefficient
of kinetic friction between the block and the incline surface is $\mu_k$. Find the
magnitude of the acceleration of the block on the inclined plane, $a_{x1}$. Express
your answers in terms of some or all of the variables $m_1$, $m_2$, $\theta$, $\mu_k$ and the
acceleration of gravity $g$.

In [3]:
m1, m2, mu_s, theta, g, mu_k = sp.symbols(
    "m1 m2 mu_s theta g mu_k", real=True, positive=True
)

![Problem 4.1 Free Body Diagrams](../figures/PS0401A-Pulley-System.jpg)

## Solver workflow summary

This section defines helper functions for the pulley equilibrium analysis:

- `latex_text(text)` escapes plain text for safe rendering in LaTeX/KaTeX, especially handling underscores like `m_1` and `m_2`.
- `pulley_force_vectors(friction_symbol, friction_direction)` builds the total force vectors for the incline block and hanging mass, using the friction symbol and direction.
- `solve_pulley_equilibrium(label, friction_symbol, friction_direction)` assembles the equilibrium equations, solves for `T`, `N1P`, and `m_2`, and displays the resulting equation and tension.
- `display_equilibrium_results(solution, friction_symbol, label)` formats and displays the normal force, no-slip condition, threshold mass, and valid range of `m_2`.

These helpers let the notebook reuse the same equilibrium logic for different friction/direction cases cleanly.


In [4]:
T, fs, fk, N1P = sp.symbols("T f_s f_k N_1P", real=True, positive=True)


def latex_text(text):
    """Escape plain text for LaTeX \text{} and KaTeX/MathJax rendering.

    This helper wraps a normal string inside a LaTeX \text{...} command and
    escapes underscores so labels such as m_1 and m_2 do not trigger LaTeX
    subscript syntax.

    Without this, any underscore in the label would be parsed as a subscript
    token, which can cause KaTeX/MathJax parse errors when used inside normal
    text labels.

    Parameters
    ----------
    text : str
        The plain text label to render inside LaTeX text mode.

    Returns
    -------
    str
        The LaTeX-safe string, e.g. "\\text{m\\_1 moving up the incline}".
    """
    return r"\text{" + text.replace("_", r"\_") + "}"


def pulley_force_vectors(friction_symbol, friction_direction):
    """Build the force vectors for the pulley system.

    This function constructs the two force balance vectors needed for
    equilibrium: one for the block on the incline and one for the hanging mass.
    It uses the supplied friction symbol and direction to model the friction
    force on the incline block.

    Parameters
    ----------
    friction_symbol : sympy.Symbol
        Symbol representing the magnitude of friction (static or kinetic).
    friction_direction : int
        Direction multiplier for friction on the incline block.
        Use -1 when friction acts opposite the positive x direction and +1
        when it acts in the positive x direction.

    Returns
    -------
    N : ReferenceFrame
        The inertial reference frame.
    P : ReferenceFrame
        The frame aligned with the incline.
    f_total_m1 : sympy expression
        The total force on mass m1 expressed in frame P.
    f_total_m2 : sympy expression
        The total force on mass m2 expressed in frame N.
    """
    N = reference_frame("N")
    P = reference_frame("P")
    P.orient_axis(N, N.z, theta)

    f_total_m1 = (
        friction_direction * friction_symbol * P.x
        + N1P * P.y
        + T * P.x
        + m1 * g * (-N.y)
    )
    f_total_m2 = 2 * T * N.y + m2 * g * (-N.y)

    return N, P, f_total_m1, f_total_m2


def solve_pulley_equilibrium(label, friction_symbol, friction_direction):
    """Solve the static equilibrium equations for the pulley setup.

    This function assembles the equilibrium matrix for both masses and
    solves for the unknowns T, N1P, and m2 using SymPy. It also displays the
    label and the equilibrium equation in LaTeX form.

    Parameters
    ----------
    label : str
        A descriptive label for the current equilibrium case, rendered in LaTeX.
    friction_symbol : sympy.Symbol
        Symbol representing the friction force magnitude.
    friction_direction : int
        Direction factor for the friction term in the incline force balance.

    Returns
    -------
    dict
        A simplified mapping of solution symbols to expressions.
    """
    N, _, f_total_m1, f_total_m2 = pulley_force_vectors(
        friction_symbol, friction_direction
    )

    stack = sp.Matrix.vstack(f_total_m1.to_matrix(N), f_total_m2.to_matrix(N))
    equilibrium = sp.Eq(stack, sp.zeros(stack.shape[0], 1))

    display(Math(latex_text(label)))
    display(Math(sp.latex(equilibrium)))

    solution = sp.solve(equilibrium, [T, N1P, m2], dict=True)[0]
    solution = {k: sp.simplify(v) for k, v in solution.items()}

    display(Math(rf"\text{{Tension in string}}: {sp.latex(solution[T])}"))
    return solution


def display_equilibrium_results(solution, friction_symbol, label):
    """Format and display the equilibrium results with LaTeX annotations.

    This function takes the solution dictionary from the equilibrium solver
    and displays the normal force, no-slip condition, critical m2 mass, and
    the valid m2 range in a human-readable LaTeX format.

    Parameters
    ----------
    solution : dict
        Mapping of solved symbols to simplified expressions.
    friction_symbol : sympy.Symbol
        Symbol used for friction in the no-slip inequality.
    label : str
        A descriptive label for the result range, rendered in LaTeX.
    """
    display(
        Math(
            rf"\text{{Normal force on }} m_1 \text{{ from pulley}}: "
            rf"{sp.latex(solution[N1P])}"
        )
    )

    no_slip_condition = sp.Lt(friction_symbol, mu_s * solution[N1P])
    display(Math(rf"\text{{No-slip condition}}: {sp.latex(no_slip_condition)}"))

    m2_no_slipping = solution[m2]
    display(
        Math(
            rf"\text{{Mass of }} m_2 \text{{ that will cause }} "
            rf"m_1 \text{{ not to slip}}: {sp.latex(m2_no_slipping)}"
        )
    )

    max_friction = mu_s * solution[N1P]
    m2_range = sp.Le(m2, m2_no_slipping.subs(friction_symbol, max_friction).factor())
    display(Math(rf"{latex_text(label)}:\boxed{{ {sp.latex(m2_range)}}}"))

In [5]:
# Compute the static threshold masses for the pulley problem

display(Markdown(r"### a.Block pulled up inclined plane"))
solution_up = solve_pulley_equilibrium(
    r"m_2,min: block 1 just barely slides up",
    fs,
    -1,
)
m2_min = sp.simplify(solution_up[m2].subs(fs, mu_s * solution_up[N1P]))
display(Math(rf"m_{{2,\min}} = {sp.latex(m2_min)}"))

display(Markdown(r"---"))
display(Markdown(r"### b. Block pulled slides down inclined plane"))
solution_down = solve_pulley_equilibrium(
    r"m_2,max: block 1 just barely slides down",
    fs,
    1,
)
m2_max = sp.simplify(solution_down[m2].subs(fs, mu_s * solution_down[N1P]))
display(Math(rf"m_{{2,\max}} = {sp.latex(m2_max)}"))


### a.Block pulled up inclined plane

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

### b. Block pulled slides down inclined plane

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [6]:
display_equilibrium_results(solution_up, fs, "m_2,min: mass for block 1 to slide up")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [7]:
display_equilibrium_results(solution_down, fs, "m_2,max: mass for block 1 to slide down")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

![](../figures/PS0401D.jpg)

In [8]:
#c.
T, fs, fk, N1P = sp.symbols("T f_s f_k N_1P", real=True, positive=True)
t = dynamicsymbols._t
N = reference_frame("N")
P = reference_frame("P")
P.orient_axis(N, N.z, sp.pi+theta)

In [9]:
m, q, ell = sp.symbols("m q ell", real=True, positive=True)
r1, r2 = dynamicsymbols("r1 r2")

length_constraint = sp.Eq(ell, r1 + ((r2 - m) + r2))
acc_constraint = sp.simplify(
    sp.Eq(
        sp.diff(length_constraint.lhs, t, 2),
        sp.diff(length_constraint.rhs, t, 2),
    )
)

display(
    Math(
        rf"\begin{{aligned}}"
            rf"\text{{length constraint: }} \quad& {spv.vlatex(length_constraint)} \\"
            rf"\text{{acceleration constraint: }} \quad& {spv.vlatex(acc_constraint)}"
        rf"\end{{aligned}}")
)


<IPython.core.display.Math object>

In [10]:
# m1 kinetics
vec_r1 = r1 * (P.x)
vec_v1 = vec_r1.dt(N)
vec_a1 = vec_v1.dt(N)

display(Math(rf"\text{{position of }} m_1: {spv.vlatex(vec_r1)}"))
display(Math(rf"\text{{velocity of }} m_1: {spv.vlatex(vec_v1)}"))
display(Math(rf"\text{{acceleration of }} m_1: {spv.vlatex(vec_a1)}"))

f_total_m1 = fk * (P.x) + N1P * (-P.y) + T * (-P.x) + m1 * g * (-N.y)
display(Math(rf"\text{{total force on }} m_1: {spv.vlatex(f_total_m1)}"))
N2Leqn1 = sp.Eq(f_total_m1.to_matrix(N), m1 * vec_a1.to_matrix(N))
display(Math(rf"\text{{Newton's 2nd Law for }} m_1: {spv.vlatex(N2Leqn1)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [11]:
# m2 kinetics
vec_r2 = m * (N.y) + q * (N.x) + r2 * (-N.y)
vec_v2 = vec_r2.dt(N)
vec_a2 = vec_v2.dt(N)
display(Math(rf"\text{{position of }} m_2: {spv.vlatex(vec_r2)}"))
display(Math(rf"\text{{velocity of }} m_2: {spv.vlatex(vec_v2)}"))
display(Math(rf"\text{{acceleration of }} m_2: {spv.vlatex(vec_a2)}"))

f_total_m2 = 2 * T * (N.y) + m2 * g * (-N.y)
display(Math(rf"\text{{total force on }} m_2: {spv.vlatex(f_total_m2)}"))

N2Leqn2 = sp.Eq(f_total_m2.to_matrix(N), m2 * vec_a2.to_matrix(N))
display(Math(rf"\text{{Newton's 2nd Law for }} m_2: {spv.vlatex(N2Leqn2)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [12]:
display(Math(rf"\text{{acceleration constraint: }} {spv.vlatex(acc_constraint)}"))
# Stack all scalar equations and include kinetic friction closure fk = mu_k N1P
stack_lhs = sp.Matrix.vstack(
    sp.Matrix([[acc_constraint.lhs]]),
    N2Leqn1.lhs,
    N2Leqn2.lhs,
)
stack_rhs = sp.Matrix.vstack(
    sp.Matrix([[acc_constraint.rhs]]),
    N2Leqn1.rhs,
    N2Leqn2.rhs,
)
stack = sp.Eq(stack_lhs, stack_rhs)
display(Math(rf"\text{{stacked Newton equations}}: {spv.vlatex(stack)}"))

eqs = list(stack_lhs - stack_rhs)
eqs.append(fk - mu_k * N1P)

unknowns = [r1.diff(t, 2), r2.diff(t, 2), T, N1P, fk]
results = sp.solve(eqs, unknowns, dict=True)
results

<IPython.core.display.Math object>

<IPython.core.display.Math object>

⎡⎧                                                                             ↪
⎢⎪         g⋅m₁⋅cos(θ)                          g⋅m₁⋅m₂⋅μₖ⋅cos(θ)              ↪
⎢⎨N_1P: ─────────────────, T: ──────────────────────────────────────────────── ↪
⎢⎪         2         2                2              2            2            ↪
⎣⎩      sin (θ) + cos (θ)     4⋅m₁⋅sin (θ) + 4⋅m₁⋅cos (θ) + m₂⋅sin (θ) + m₂⋅co ↪

↪                                        2                                     ↪
↪                           2⋅g⋅m₁⋅m₂⋅sin (θ)                                  ↪
↪ ───── + ───────────────────────────────────────────────────── + ──────────── ↪
↪  2              2              2            2            2              2    ↪
↪ s (θ)   4⋅m₁⋅sin (θ) + 4⋅m₁⋅cos (θ) + m₂⋅sin (θ) + m₂⋅cos (θ)   4⋅m₁⋅sin (θ) ↪

↪                                                                            2 ↪
↪        g⋅m₁⋅m₂⋅sin(θ)                                         2⋅g⋅m₁⋅m₂⋅cos  ↪
↪ ────────────────────────

In [13]:
results = {k: sp.simplify(v) for k, v in results[0].items()}
display(Math(rf"\text{{results}}: {spv.vlatex(results)}"))

display(Math(rf"\text{{Tension}}: {spv.vlatex(results[T])}"))

acceleration_m1 = results[r1.diff(t, 2)].factor()
display(Math(rf"\text{{Acceleration of }} m_1: {spv.vlatex(acceleration_m1)}"))

acceleration_m2 = results[r2.diff(t, 2)].factor()
display(Math(rf"\text{{Acceleration of }} m_2: {spv.vlatex(acceleration_m2)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [14]:
# Compact symbolic sanity checks for Problem 4.1
checks = {}

# Dynamic acceleration from part (c)
a1_expr = sp.simplify(acceleration_m1)

# 1) Zero-acceleration boundary
checks["a1_zero_boundary"] = sp.simplify(
    a1_expr.subs(m2, 2 * m1 * (sp.sin(theta) + mu_k * sp.cos(theta)))
)

# 2) Frictionless limit (mu_k -> 0)
checks["a1_mu_k_to_0"] = sp.simplify(a1_expr.subs(mu_k, 0))

# 3) Heavy hanging mass limit
checks["a1_m2_to_inf"] = sp.simplify(sp.limit(a1_expr, m2, sp.oo))

# 4) No hanging mass
checks["a1_m2_to_0"] = sp.simplify(a1_expr.subs(m2, 0))

# 5) Static threshold spacing identity
checks["m2_gap_identity"] = sp.simplify(m2_min - m2_max - 4 * m1 * mu_s * sp.cos(theta))

# 6) Zero-static-friction collapse
checks["mu_s_zero_collapse"] = sp.simplify((m2_min - m2_max).subs(mu_s, 0))

# Display expected targets for quick visual confirmation
targets = {
    "a1_zero_boundary": 0,
    "a1_mu_k_to_0": sp.simplify(2 * g * (2 * m1 * sp.sin(theta) - m2) / (4 * m1 + m2)),
    "a1_m2_to_inf": -2 * g,
    "a1_m2_to_0": sp.simplify(g * (sp.sin(theta) + mu_k * sp.cos(theta))),
    "m2_gap_identity": 0,
    "mu_s_zero_collapse": 0,
}

rows = []
all_pass = True
for name, value in checks.items():
    ok = sp.simplify(value - targets[name]) == 0
    all_pass = all_pass and ok
    rows.append((name, sp.simplify(value), targets[name], ok))

display(Markdown("### Compact Sanity Checks"))
for name, value, target, ok in rows:
    safe_name = name.replace("_", r"\_")
    status = "PASS" if ok else "FAIL"
    display(Math(rf"\text{{{safe_name}: {status}}}"))
    display(Math(rf"\text{{value}} = {sp.latex(value)}"))
    display(Math(rf"\text{{target}} = {sp.latex(target)}"))

display(Math(rf"\text{{All checks pass}} = {sp.latex(all_pass)}"))

### Compact Sanity Checks

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>